In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Find project root
cwd = Path.cwd()

if (cwd / "data").is_dir() and (cwd / "outputs").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").is_dir() and (cwd.parent / "outputs").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find the HAALSI_Project data/ and outputs/ folders."
    )

DATA = PROJECT_ROOT / "data"
OUTPUTS = PROJECT_ROOT / "outputs"

FIGURES = PROJECT_ROOT / "figures" / "supervisor"
FIGURES.mkdir(parents=True, exist_ok=True)

# Load datasets from data/ and results from outputs/
def load_csv(prefix):
    for folder in [DATA, OUTPUTS]:
        matches = sorted(
            folder.glob(f"{prefix}*.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True
        )

        if matches:
            df = pd.read_csv(matches[0])
            print(
                f"Loaded [{folder.name}]: {matches[0].name} "
                f"({df.shape[0]:,} rows × {df.shape[1]} columns)"
            )
            return df

    print(f"Not found: {prefix}")
    return None


def get_col(df, *names):
    if df is None:
        return None

    lookup = {str(c).lower(): c for c in df.columns}

    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    return None


def save_figure(name):
    plt.tight_layout()
    plt.savefig(
        FIGURES / f"{name}.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()


print("Project root:", PROJECT_ROOT)
print("Datasets in data/:", len(list(DATA.glob("*.csv"))))
print("Results in outputs/:", len(list(OUTPUTS.glob("*.csv"))))

In [ ]:
primary = load_csv("section3_7_model_comparison")
controlled = load_csv("section3_8_model_summary")
ablation = load_csv("section3_4_ablation_performance")
shap = load_csv("section3_6_shap_importance")

primary_bootstrap = load_csv(
    "section3_7_paired_model_comparison_bootstrap"
)

education_bootstrap = load_csv(
    "section3_4_education_ablation_bootstrap"
)

for name, result in {
    "Primary comparison": primary,
    "Controlled comparison": controlled,
    "Education ablation": ablation,
    "SHAP": shap
}.items():
    print(f"\n{name}")
    if result is not None:
        display(result.head())
        print("Columns:", result.columns.tolist())

In [ ]:
def plot_model_performance(df, title, filename):
    if df is None or df.empty:
        print(f"No data available for {title}")
        return

    required = ["Model", "FeatureSet", "MAE_mean"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        print(f"Missing columns: {missing}")
        print("Available:", df.columns.tolist())
        return

    plot_df = df.copy()
    plot_df["MAE_mean"] = pd.to_numeric(
        plot_df["MAE_mean"], errors="coerce"
    )
    plot_df = plot_df.dropna(subset=["MAE_mean"])

    fig, ax = plt.subplots(figsize=(12, 6))

    sns.barplot(
        data=plot_df,
        x="Model",
        y="MAE_mean",
        hue="FeatureSet",
        errorbar=None,
        ax=ax
    )

    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel("Mean absolute error (MAE)")
    ax.legend(title="Feature representation",
              bbox_to_anchor=(1.02, 1), loc="upper left")

    save_figure(filename)

In [ ]:
plot_model_performance(
    primary,
    "Primary Prediction Results: Total vs Decomposed Recall",
    "08_primary_model_comparison"
)

In [ ]:
plot_model_performance(
    ablation,
    "Prediction Performance With and Without Education",
    "10_education_ablation"
)

In [ ]:
if ablation is not None:
    plot_df = ablation.copy()

    g = sns.catplot(
        data=plot_df,
        x="FeatureSet",
        y="MAE_mean",
        hue="EducationMode",
        col="Model",
        kind="bar",
        errorbar=None,
        col_wrap=2,
        height=4.5,
        aspect=1.2,
        sharey=True
    )

    g.set_axis_labels("Feature representation", "MAE")
    g.set_titles("{col_name}")

    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=35)

    g.figure.suptitle(
        "Education Ablation: Prediction Error by Model",
        y=1.04,
        fontsize=15
    )

    g.figure.savefig(
        FIGURES / "10_education_ablation.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
if primary_bootstrap is not None:
    print("Primary model comparison: bootstrap uncertainty")
    display(primary_bootstrap)

if education_bootstrap is not None:
    print("Education ablation: bootstrap uncertainty")
    display(education_bootstrap)

In [ ]:
if shap is not None:
    print("SHAP columns:", shap.columns.tolist())

    def match_column(df, candidates):
        normalized = {
            str(c).lower().replace("_", "").replace(" ", ""): c
            for c in df.columns
        }
        for candidate in candidates:
            key = candidate.lower().replace("_", "").replace(" ", "")
            if key in normalized:
                return normalized[key]
        return None

    feature_col = match_column(shap, [
        "Feature", "FeatureName", "Variable"
    ])
    value_col = match_column(shap, [
        "MeanAbsSHAP", "MeanAbsoluteSHAP",
        "MeanSHAP", "SHAPImportance",
        "Importance", "MeanAbsSHAPValue"
    ])
    model_col = match_column(shap, [
        "Model", "ModelName"
    ])

    if feature_col and value_col:
        shap_plot = shap.copy()
        shap_plot[value_col] = pd.to_numeric(
            shap_plot[value_col], errors="coerce"
        )

        models = (
            shap_plot[model_col].dropna().unique()
            if model_col else ["All"]
        )

        for model in models:
            subset = (
                shap_plot[shap_plot[model_col] == model]
                if model_col else shap_plot
            )

            subset = subset.nlargest(12, value_col)

            fig, ax = plt.subplots(figsize=(9, 6))
            sns.barplot(
                data=subset,
                x=value_col,
                y=feature_col,
                color="#4C78A8",
                ax=ax
            )

            ax.set_title(f"SHAP Importance — {model}")
            ax.set_xlabel("Mean absolute SHAP value")
            ax.set_ylabel("Feature")

            safe_name = str(model).replace(" ", "_")
            save_figure(f"11_shap_{safe_name}")
    else:
        print("Unrecognised SHAP columns:")
        print(shap.columns.tolist())

In [ ]:
if primary_bootstrap is not None:
    boot = primary_bootstrap.copy()

    boot["Comparison"] = (
        boot["Model"] + ": "
        + boot["Baseline"] + " vs "
        + boot["Proposed"]
    )

    boot = boot.sort_values("Improvement_PositiveIsBetter")

    y = np.arange(len(boot))
    estimates = boot["Improvement_PositiveIsBetter"].to_numpy()
    lower = boot["CI95_Lower"].to_numpy()
    upper = boot["CI95_Upper"].to_numpy()

    fig, ax = plt.subplots(figsize=(11, 6))

    ax.errorbar(
        estimates,
        y,
        xerr=[estimates - lower, upper - estimates],
        fmt="o",
        capsize=4,
        color="#3274A1"
    )

    ax.axvline(0, color="black", linestyle="--", alpha=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(boot["Comparison"])
    ax.set_xlabel(
        "MAE improvement: positive favours decomposed recall"
    )
    ax.set_title(
        "Total vs Decomposed Recall: Bootstrap 95% CIs"
    )

    save_figure("12_primary_bootstrap_comparison")